# Fast reads of remote NetCDF: eager reader + cached store

Demo to identify speed up potential of using cached store + eager reader. 

Reading a NetCDF over HTTPS is slow for one reason: h5py walks the HDF5 metadata with
many small, scattered, *sequentially dependent* reads. Over a ~250 ms link each one is
a separate round-trip. The default reader also discards its buffer on every seek, so a
649 MiB CMIP6 file costs ~648 separate range GETs.

Two `obspec-utils` pieces fix it:

- **`EagerStoreReader`** — fetch the object with concurrent range requests, then let
  h5py parse from memory. Trades bytes for round-trips.
- **`CachingReadableStore`** — fetch each object once and serve every later range from
  RAM. This matters more than it looks: loading the `time` coordinate costs **12,606
  range requests**, because `time` is chunked one chunk per timestep.

Measured on the file below (CEDA, 649 MiB, ~11 MB/s link):

| | default | eager + cached |
|---|---:|---:|
| metadata walk | 346 s | 59 s |
| loading coordinates | 302 s | 1.2 s |
| **end to end** | **414 s** | **58 s** |

In [1]:
import time

import xarray as xr
from obspec_utils.readers import EagerStoreReader
from obspec_utils.wrappers import CachingReadableStore
from obstore.store import from_url

HOST = "https://dap.ceda.ac.uk"
PATH = (
    "badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/"
    "AERday/zg500/gn/v20190726/"
    "zg500_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20150101-20491230.nc"
)
URL = f"{HOST}/{PATH}"

# One cached store, reused everywhere below. max_size must exceed the file size,
# or the object is evicted and refetched.
store = CachingReadableStore(from_url(HOST), max_size=2 * 1024**3)

## 1. `xr.open_dataset`

`EagerStoreReader` implements the `ReadableFile` protocol (`read`/`seek`/`tell`), which
is exactly what `h5netcdf` wants — so it substitutes for the URL string with no other
changes.

In [2]:
t = time.perf_counter()
ds = xr.open_dataset(EagerStoreReader(store, PATH), engine="h5netcdf")
print(f"{time.perf_counter() - t:.1f}s")
ds

48.1s


<xarray.Dataset> Size: 1GB
Dimensions:    (time: 12600, bnds: 2, lat: 144, lon: 192)
Coordinates:
  * time       (time) object 101kB 2015-01-01 12:00:00 ... 2049-12-30 12:00:00
  * lat        (lat) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * lon        (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
    plev       float64 8B ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 202kB ...
    lat_bnds   (lat, bnds) float64 2kB ...
    lon_bnds   (lon, bnds) float64 3kB ...
    zg500      (time, lat, lon) float32 1GB ...
Attributes: (12/46)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   59400.0
    branch_time_in_parent:  59400.0
    creation_date:          2019-07-24T11:00:55Z
    ...                     ...
    title:                  UKESM1-0-LL output prepared for CMIP6
    tracking_id:            hdl:21.14100/67d4bba1-c060-414a-820f-febe2dbfc694
    variable_id:            zg500
    variant_label:          r1i1p1f2
    license:                CMIP6 model data produced by the Met Office Hadle...
    cmor_version:           3.4.0

## 2. `open_virtual_dataset`

virtualizarr's `HDFParser` hardcodes its own reader, so swapping in the eager one means
a small parser of our own. A parser is just a callable `(url, registry) -> ManifestStore`.

Pass the **same cached store** in the registry — that is what makes the coordinate load
cheap.

In [3]:
from virtualizarr.manifests import ManifestStore
from virtualizarr.parsers.hdf.hdf import _construct_manifest_group
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.xarray import open_virtual_dataset


class EagerHDFParser:
    """HDFParser, but reading through an EagerStoreReader."""

    def __call__(self, url, registry):
        store, path = registry.resolve(url)
        reader = EagerStoreReader(store, path)
        try:
            group = _construct_manifest_group(filepath=url, reader=reader)
        finally:
            reader.close()
        return ManifestStore(group, registry=registry)


registry = ObjectStoreRegistry({HOST: store})

t = time.perf_counter()
vds = open_virtual_dataset(url=URL, registry=registry, parser=EagerHDFParser())
print(f"{time.perf_counter() - t:.1f}s")
vds

/Users/juliusbusecke/Code/cmip7-virtualization/.worktrees/parsing-performance/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


1.9s


<xarray.Dataset> Size: 1GB
Dimensions:    (time: 12600, lat: 144, lon: 192, bnds: 2)
Coordinates:
  * time       (time) object 101kB 2015-01-01 12:00:00 ... 2049-12-30 12:00:00
  * lat        (lat) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * lon        (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
    plev       float64 8B ManifestArray<shape=(), dtype=float64, chunks=()>
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) float64 202kB ManifestArray<shape=(12600, 2), dty...
    lat_bnds   (lat, bnds) float64 2kB ManifestArray<shape=(144, 2), dtype=fl...
    lon_bnds   (lon, bnds) float64 3kB ManifestArray<shape=(192, 2), dtype=fl...
    zg500      (time, lat, lon) float32 1GB ManifestArray<shape=(12600, 144, ...
Attributes: (12/46)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   59400.0
    branch_time_in_parent:  59400.0
    creation_date:          2019-07-24T11:00:55Z
    ...                     ...
    title:                  UKESM1-0-LL output prepared for CMIP6
    tracking_id:            hdl:21.14100/67d4bba1-c060-414a-820f-febe2dbfc694
    variable_id:            zg500
    variant_label:          r1i1p1f2
    license:                CMIP6 model data produced by the Met Office Hadle...
    cmor_version:           3.4.0

That second cell is fast because the cache already holds the file from cell 1. Build a
fresh `CachingReadableStore` to time it cold.

## The baseline, for comparison

Uncached store + stock parser. Takes ~6 minutes — run it only if you want to see the
difference yourself.

In [4]:
from virtualizarr.parsers import HDFParser

plain = ObjectStoreRegistry({HOST: from_url(HOST)})

t = time.perf_counter()
slow = open_virtual_dataset(url=URL, registry=plain, parser=HDFParser())
print(f"{time.perf_counter() - t:.1f}s")

/Users/juliusbusecke/Code/cmip7-virtualization/.worktrees/parsing-performance/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


748.2s
